<a href="https://colab.research.google.com/github/404AliOnFire/aib-chatbot/blob/main/Qa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Install the required libraries
print("... Installing necessary libraries ...")
!pip install transformers datasets torch sentence-transformers -q

# 2. Mount Google Drive
from google.colab import drive

drive.mount('/content/drive')
print("... Google Drive connected successfully ...")

# 3. Define paths and variables
model_path = "/content/drive/MyDrive/arabert-finetuned-aib-qa"
data_file_path = "/content/drive/MyDrive/QA/QA.json"

# 4. Load the trained model and tokenizer
from transformers import AutoTokenizer, AutoModelForQuestionAnswering
import torch

print("... Loading trained model ...")

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForQuestionAnswering.from_pretrained(model_path)

# 5. Load the data, including the original contexts and questions
import json

print("... Loading training data ...")

with open(data_file_path, "r", encoding="utf-8") as f:
    qa_data = json.load(f)

contexts = [item["context"] for item in qa_data]
questions = [item["question"] for item in qa_data]

# 6. Generate embeddings for all contexts using sentence-transformers
from sentence_transformers import SentenceTransformer, util

print("... Embedding all contexts. This may take some time ...")

embedder = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

context_embeddings = embedder.encode(
    contexts,
    convert_to_tensor=True
)

In [ ]:


# 0. Install libraries
print("... Installing libraries ...")
!pip install -q transformers datasets accelerate evaluate sentencepiece

# 1. Mount Google Drive
from google.colab import drive

drive.mount('/content/drive')

# 2. Basic variables
model_name = "aubmindlab/bert-base-arabertv2"
data_file_path = "/content/drive/MyDrive/QA/QA.json"
new_model_name = "arabert-finetuned-aib-qa"

# 3. Import required libraries
import collections
import numpy as np
from datasets import load_dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForQuestionAnswering,
    TrainingArguments,
    Trainer,
    default_data_collator
)
import evaluate
import json
import os
from tqdm import tqdm

# 4. Load the dataset
# This assumes the JSON file is already in a SQuAD-like format
print("... Loading dataset ...")
raw_dataset = load_dataset("json", data_files=data_file_path, split="train")

print(raw_dataset)

# 5. Split the dataset into training and evaluation sets
split_dataset = raw_dataset.train_test_split(test_size=0.1, seed=42)
print("... Dataset split completed: ", split_dataset)

# 6. Load the tokenizer and model
print(f"... Loading tokenizer and model: {model_name} ...")

tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
model = AutoModelForQuestionAnswering.from_pretrained(model_name)

# 7. Prepare features for training
# This handles long contexts using overflow and stride
max_length = 384
doc_stride = 128


def prepare_train_features(examples):
    questions = [q.strip() for q in examples["question"]]
    contexts = [c.strip() for c in examples["context"]]
    answers = examples["answers"]

    tokenized_examples = tokenizer(
        questions,
        contexts,
        truncation="only_second",
        max_length=max_length,
        stride=doc_stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    sample_mapping = tokenized_examples["overflow_to_sample_mapping"]
    offset_mapping = tokenized_examples["offset_mapping"]

    start_positions = []
    end_positions = []

    for i, offsets in enumerate(offset_mapping):
        input_ids = tokenized_examples["input_ids"][i]
        cls_index = input_ids.index(tokenizer.cls_token_id)

        sequence_ids = tokenized_examples.sequence_ids(i)
        sample_index = sample_mapping[i]
        answer = answers[sample_index]

        if len(answer["text"]) == 0:
            start_positions.append(cls_index)
            end_positions.append(cls_index)
            continue

        start_char = answer["answer_start"][0]
        end_char = start_char + len(answer["text"][0])

        token_start_index = 0
        while sequence_ids[token_start_index] != 1:
            token_start_index += 1

        token_end_index = len(input_ids) - 1
        while sequence_ids[token_end_index] != 1:
            token_end_index -= 1

        if (
            offsets[token_start_index][0] > start_char
            or offsets[token_end_index][1] < end_char
        ):
            start_positions.append(cls_index)
            end_positions.append(cls_index)
        else:
            idx = token_start_index

            while idx <= token_end_index and offsets[idx][0] <= start_char:
                idx += 1

            start_positions.append(idx - 1)

            idx = token_end_index

            while idx >= token_start_index and offsets[idx][1] >= end_char:
                idx -= 1

            end_positions.append(idx + 1)

    tokenized_examples["start_positions"] = start_positions
    tokenized_examples["end_positions"] = end_positions

    return tokenized_examples


# 8. Tokenize the dataset
print("... Tokenizing dataset. This may take some time ...")

tokenized_datasets = split_dataset.map(
    prepare_train_features,
    batched=True,
    remove_columns=split_dataset["train"].column_names
)

# 9. Post-process model outputs into text predictions
def postprocess_qa_predictions(
    examples,
    features,
    raw_predictions,
    n_best_size=20,
    max_answer_length=30
):
    all_start_logits, all_end_logits = raw_predictions

    example_id_to_index = (
        {k: i for i, k in enumerate(examples["id"])}
        if "id" in examples.column_names
        else {i: i for i in range(len(examples))}
    )

    features_per_example = collections.defaultdict(list)

    for i, feature in enumerate(features):
        sample_index = (
            feature["example_index"]
            if "example_index" in feature
            else feature.get("overflow_to_sample_mapping", i)
        )

        features_per_example[sample_index].append(i)

    predictions = collections.OrderedDict()

    for example_index, example in enumerate(tqdm(examples)):
        feature_indices = features_per_example[example_index]
        prelim_predictions = []

        # Collect candidate answers from all features related to the same example
        for fi in feature_indices:
            start_logits = all_start_logits[fi]
            end_logits = all_end_logits[fi]
            offset_mapping = features[fi]["offset_mapping"]

            # Find the best start and end token candidates
            start_indexes = np.argsort(start_logits)[-1: -n_best_size - 1: -1].tolist()
            end_indexes = np.argsort(end_logits)[-1: -n_best_size - 1: -1].tolist()

            for start_index in start_indexes:
                for end_index in end_indexes:
                    if start_index >= len(offset_mapping) or end_index >= len(offset_mapping):
                        continue

                    if offset_mapping[start_index] is None or offset_mapping[end_index] is None:
                        continue

                    if end_index < start_index:
                        continue

                    length = offset_mapping[end_index][1] - offset_mapping[start_index][0]

                    if length > max_answer_length:
                        continue

                    start_char = offset_mapping[start_index][0]
                    end_char = offset_mapping[end_index][1]
                    text = example["context"][start_char:end_char]
                    score = start_logits[start_index] + end_logits[end_index]

                    prelim_predictions.append({
                        "score": float(score),
                        "text": text
                    })

        if len(prelim_predictions) == 0:
            predictions[example_index] = ""
            continue

        # Keep the best candidate answer
        best = sorted(
            prelim_predictions,
            key=lambda x: x["score"],
            reverse=True
        )[:n_best_size]

        predictions[example_index] = best[0]["text"]

    # Return predictions in the same order as the examples
    return [
        predictions[i] if i in predictions else ""
        for i in range(len(examples))
    ]


# 10. Load the metric and define the evaluation function
metric = evaluate.load("squad")


def compute_metrics_eval(eval_pred):
    start_logits, end_logits = eval_pred.predictions
    features = tokenized_datasets["test"]
    examples = split_dataset["test"]

    # Convert logits into text answers
    predicted_texts = postprocess_qa_predictions(
        examples=examples,
        features=features,
        raw_predictions=(start_logits, end_logits)
    )

    references = []
    predictions_for_metric = []

    for i, ex in enumerate(examples):
        # Use the original ID and make sure it is a string
        example_id = ex.get("id", str(i))

        references.append({
            "id": str(example_id),
            "answers": {
                "text": ex["answers"]["text"],
                "answer_start": ex["answers"]["answer_start"]
            }
        })

        predictions_for_metric.append({
            "id": str(example_id),
            "prediction_text": predicted_texts[i]
        })

    return metric.compute(
        predictions=predictions_for_metric,
        references=references
    )


# 11. Define training arguments
training_args = TrainingArguments(
    output_dir=new_model_name,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=6,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=2,
    fp16=True,
    push_to_hub=False,
    logging_steps=50,
)

# 12. Create the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    tokenizer=tokenizer,
    data_collator=default_data_collator,
    compute_metrics=compute_metrics_eval
)

# 13. Start training
print("... Starting training ...")
trainer.train()

# 14. Save the best model to Google Drive
final_model_path = f"/content/drive/MyDrive/{new_model_name}"

os.makedirs(final_model_path, exist_ok=True)

trainer.save_model(final_model_path)
tokenizer.save_pretrained(final_model_path)

print(f"Model saved to: {final_model_path}")

print("... Done. Training finished and the model was exported successfully.")

In [ ]:
# 1. Install libraries
# transformers and datasets were already installed during training.
# Here, we install sentence-transformers, which is used for text similarity search.
print("... Installing libraries ...")
!pip install -q sentence-transformers

# 2. Import the required libraries
from google.colab import drive
from transformers import pipeline
from sentence_transformers import SentenceTransformer, util
import json
import torch  # Core library for working with vectors and tensors

# 3. Mount Google Drive
print("... Mounting Google Drive ...")
drive.mount('/content/drive')

In [ ]:
# 1. Define paths
# Make sure to adjust them based on your files
data_file = "/content/drive/MyDrive/QA/QA.json"
trained_model_path = "/content/drive/MyDrive/arabert-finetuned-aib-qa"

# 2. Load the data and prepare it for both search methods
print("... Loading and preparing the knowledge base ...")

all_contexts = []  # Unique list of all context paragraphs
all_questions = []  # List of all training questions
question_to_context_map = []  # Stores the context related to each training question

with open(data_file, "r", encoding="utf-8") as f:
    data = json.load(f)

    # Temporary dictionary to avoid duplicate contexts
    unique_contexts = {}

    for item in data:
        context = item["context"]
        question = item["question"]

        all_questions.append(question)
        question_to_context_map.append(context)

        if context not in unique_contexts:
            unique_contexts[context] = True
            all_contexts.append(context)

print(f"Loaded {len(all_contexts)} unique contexts and {len(all_questions)} training questions.")

# 3. Load the retriever model
# The retriever converts text into vectors/embeddings
print("... Loading the retriever model: SentenceTransformer ...")

retriever_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

# 4. Build two indexes: one for contexts and one for questions
# This process may take one or two minutes
print("... Encoding contexts into vectors ...")

context_embeddings = retriever_model.encode(
    all_contexts,
    convert_to_tensor=True,
    show_progress_bar=True
)

print("... Encoding training questions into vectors ...")

question_embeddings = retriever_model.encode(
    all_questions,
    convert_to_tensor=True,
    show_progress_bar=True
)

print("Hybrid retriever is ready.")

In [ ]:
print("... Loading your fine-tuned Reader model (QA Pipeline) ...")
reader_pipeline = pipeline(
    "question-answering",
    model=trained_model_path,
    tokenizer=trained_model_path
)
print("النموذج القارئ جاهز!")

In [ ]:
def ask_hybrid_chatbot(user_question, alpha=0.5, top_k=5):
    import pandas as pd

    """
    Chatbot that uses hybrid retrieval.

    - alpha: the weight given to context search, between 0 and 1.
             alpha=1.0 means using only context search.
             alpha=0.0 means using only question search.
             alpha=0.5 means a balance between both.

    - top_k: the number of top results to check from each search method.
    """

    print(f"New question: {user_question}")
    print(f"Using alpha = {alpha}, which is the context search weight.")

    question_embedding = retriever_model.encode(
        user_question,
        convert_to_tensor=True
    )

    # 1. Context search
    context_hits = util.semantic_search(
        question_embedding,
        context_embeddings,
        top_k=top_k
    )[0]

    context_scores = {
        hit["corpus_id"]: hit["score"]
        for hit in context_hits
    }

    # 2. Question search
    question_hits = util.semantic_search(
        question_embedding,
        question_embeddings,
        top_k=top_k
    )[0]

    # 3. Merge the results and calculate the final score
    final_scores = {}
    candidate_contexts = {}  # Stores candidate contexts and their scores

    # Add context search results
    for hit in context_hits:
        ctx_idx = hit["corpus_id"]
        context_text = all_contexts[ctx_idx]

        final_scores[context_text] = hit["score"] * alpha

        candidate_contexts[context_text] = {
            "context_score": hit["score"],
            "question_score": 0
        }

    # Add question search results
    for hit in question_hits:
        q_idx = hit["corpus_id"]
        context_text = question_to_context_map[q_idx]

        # Use 1 - alpha as the weight for question search
        score = hit["score"] * (1 - alpha)

        if context_text in final_scores:
            final_scores[context_text] += score
        else:
            final_scores[context_text] = score

        # Store scores for display
        if context_text not in candidate_contexts:
            candidate_contexts[context_text] = {
                "context_score": 0,
                "question_score": 0
            }

        candidate_contexts[context_text]["question_score"] = hit["score"]

    # 4. Choose the best context
    if not final_scores:
        print("No suitable context was found.")
        return

    best_context = max(final_scores, key=final_scores.get)

    # 5. Display intermediate search results for explanation
    print("\n--- Search analysis ---")

    df = pd.DataFrame.from_dict(
        candidate_contexts,
        orient="index"
    )

    df["final_score"] = df.index.map(final_scores)
    df = df.sort_values("final_score", ascending=False)

    print(df.head())

    print("\nBest context selected based on the merged score:")
    print(best_context)

    # 6. Reading stage
    result = reader_pipeline(
        question=user_question,
        context=best_context
    )

    print("\n" + "=" * 20)
    print(f"Final answer: {result['answer']}")
    print(f"Reader confidence score: {result['score']:.4f}")
    print("=" * 20 + "\n")

In [ ]:
ask_hybrid_chatbot("", alpha=0.5)
ask_hybrid_chatbot("من هو مدير فرع يطا", alpha=1)
ask_hybrid_chatbot("وين موقع فرع الحرس؟", alpha=0.3)